In [ ]:
import pandas as pd
import numpy as np
import os
from io import StringIO
from unidecode import unidecode
from datetime import timedelta
from scipy.interpolate import PchipInterpolator
import toml
from src.feature_building_utils import *

In [ ]:
sid = 3

In [ ]:
df = pd.read_parquet(f'data/processed_data/S{sid}-approx-coordinates.parquet')
time_delta = pd.to_timedelta(df.index.astype(str))
base_date = pd.Timestamp('2022-11-11') + pd.Timedelta(days=sid-1)
df.index = base_date + time_delta

In [ ]:
docs = toml.load(TOML_PATH)

In [ ]:
wd_path = os.path.dirname('data/raw_data/RW_20250505181303_690121/')

In [ ]:
for file in os.listdir(wd_path):
    if file.endswith('.txt'):
        path = os.path.join(wd_path, file)

        # Option A: read & slice, then parse with header=0
        with open(path, 'r') as f:
            lines = f.readlines()[11:18]     # lines 5–10
        csv_block = ''.join(lines)
        df1 = pd.read_csv(StringIO(csv_block), header=0)

dictionary = df1.set_index('Id_Sensore')['Nome_Sensore'].to_dict()

In [ ]:
dfs = []

for file in os.listdir(wd_path):
    if file.endswith('.csv'):
        fn = os.path.join(wd_path, file)
        df_temp = pd.read_csv(
            fn,
            index_col='Data-Ora',
            parse_dates=['Data-Ora']
        )
        original_name = df_temp.drop('Id Sensore', axis=1).columns[0]
        id = df_temp['Id Sensore'].unique()[0]
        name = dictionary[id].strip() + ' ' + original_name.strip()
        name = name.replace(' ', '_').lower()
        # unidecode the name variable to the closest carachter dont ignore, change to the closest for example  á -> a
        name = unidecode(name)

        df_temp.rename(columns={original_name: name}, inplace=True)
        df_temp.drop('Id Sensore', axis=1, inplace=True)
        dfs.append(df_temp)

# Concatenate all dataframes
wd_df = pd.concat(dfs, axis=1)

# remove missing values
wd_df = wd_df.replace([888, 8888, -999], pd.NA)
wd_df = wd_df.replace([777, 7777], 0)


In [ ]:
#filter wd_df to show the rows where the index correspond only to the last 20 days of november
wd_df = wd_df.loc[pd.to_datetime('2022-11-11') + timedelta(days=sid-1):pd.to_datetime('2022-11-11') + timedelta(days=sid)]

In [ ]:
# 1) Make sure your index is datetime and sorted
wd_copy = wd_df.copy().sort_index()
if not isinstance(wd_copy.index, pd.DatetimeIndex):
    wd_copy.index = pd.to_datetime(wd_copy.index)

# 2) Build the new 1-second index
new_idx = pd.date_range(start=wd_copy.index[0], end=wd_copy.index[-1], freq='1S')

# 3) Precompute numeric time vectors (seconds since epoch)
x_old = wd_copy.index.astype(np.int64) / 1e9
x_new = new_idx.astype(np.int64) / 1e9

# 4) Prepare the upsampled DataFrame
df_1s = pd.DataFrame(index=new_idx)

# 5) Loop over each column and interpolate
for col in wd_copy.columns:
    y = wd_copy[col]
    mask = y.notna()
    n_obs = mask.sum()

    x_col = x_old[mask]
    y_col = y[mask].values
    pchip = PchipInterpolator(x_col, y_col)
    df_1s[col] = pchip(x_new)

In [ ]:
for col in df_1s.columns:
    df[col] = df_1s.loc[df.index, col].values

In [ ]:
feature_name = df_1s.columns[0]
print(f"Feature name: {feature_name}")

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Estimated maximum wind speed in m/s"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "weather_data.ipynb"
docs[feature_name]["source"] = "ARPA Lombardia"
docs[feature_name]["source_url"] = "https://www.arpalombardia.it/temi-ambientali/meteo-e-clima/form-richiesta-dati/"

In [ ]:
feature_name = df_1s.columns[1]
print(f"Feature name: {feature_name}")

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Estimated wind direction at maximum wind speed in degrees from North"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "weather_data.ipynb"
docs[feature_name]["source"] = "ARPA Lombardia"
docs[feature_name]["source_url"] = "https://www.arpalombardia.it/temi-ambientali/meteo-e-clima/form-richiesta-dati/"

In [ ]:
feature_name = df_1s.columns[2]
print(f"Feature name: {feature_name}")

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Estimated average wind speed in m/s"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "weather_data.ipynb"
docs[feature_name]["source"] = "ARPA Lombardia"


In [ ]:
feature_name = df_1s.columns[3]
print(f"Feature name: {feature_name}")

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Estimated average global solar radiation in W/m2"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "weather_data.ipynb"
docs[feature_name]["source"] = "ARPA Lombardia"

In [ ]:
feature_name = df_1s.columns[4]
print(f"Feature name: {feature_name}")

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Estimated Accumulated precipitation value in mm"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "weather_data.ipynb"
docs[feature_name]["source"] = "ARPA Lombardia"

In [ ]:
feature_name = df_1s.columns[5]
print(f"Feature name: {feature_name}")

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Estimated average wind direction in degrees from North"
docs[feature_name]["type"] = "float"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "weather_data.ipynb"
docs[feature_name]["source"] = "ARPA Lombardia"

In [ ]:
df[feature_name].plot()

In [ ]:
with open("documentation/feature_docs.toml", "w") as f:
    toml.dump(docs, f)

In [ ]:
df.to_parquet('data/processed_data/S3-approx-coordinates.parquet')